# Sector ML Classification - Research Notebook

## Objectif
Developper une strategie de rotation sectorielle utilisant le Machine Learning pour classer les 11 secteurs US en 3 categories:
- **BUY** (2): Fort potentiel de hausse
- **HOLD** (1): Neutre
- **AVOID** (0): Risque de baisse

## Approche
- **Features**: Inspirees de QC-Py-18 (Feature Engineering)
- **Model**: Random Forest Classifier (3 classes)
- **Rebalancing**: Mensuel sur les top-3 secteurs classes BUY
- **Training**: 2015-2022, Backtest: 2023-2025

## 11 Sector ETFs
- XLK: Technology
- XLF: Financials  
- XLV: Healthcare
- XLE: Energy
- XLY: Consumer Discretionary
- XLP: Consumer Staples
- XLI: Industrials
- XLB: Materials
- XLU: Utilities
- XLRE: Real Estate
- XLC: Communication Services

In [ ]:
# Imports et setup QuantBook
from AlgorithmImports import *
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

qb = QuantBook()

# 11 Sector ETFs
SECTOR_ETFS = {
    "XLK": "Technology",
    "XLF": "Financials",
    "XLV": "Healthcare",
    "XLE": "Energy",
    "XLY": "Consumer Discretionary",
    "XLP": "Consumer Staples",
    "XLI": "Industrials",
    "XLB": "Materials",
    "XLU": "Utilities",
    "XLRE": "Real Estate",
    "XLC": "Communication Services"
}

# Ajouter les symboles
symbols = []
for ticker in SECTOR_ETFS.keys():
    symbol = qb.add_equity(ticker, Resolution.DAILY, Market.USA).symbol
    symbols.append(symbol)

# Benchmark SPY
spy = qb.add_equity("SPY", Resolution.DAILY, Market.USA).symbol

print(f"Sectors charges: {len(symbols)}")
print(f"Benchmark: {spy}")

In [ ]:
# Chargement des donnees (Training: 2015-2022)
TRAIN_START = datetime(2015, 1, 1)
TRAIN_END = datetime(2022, 12, 31)
BACKTEST_START = datetime(2023, 1, 1)
BACKTEST_END = datetime(2025, 12, 31)

# Charger l'historique
history = qb.history(symbols, TRAIN_START, BACKTEST_END, Resolution.DAILY)

print(f"Donnees chargees: {len(history)} jours")
print(f"Periode training: {TRAIN_START} a {TRAIN_END}")
print(f"Periode backtest: {BACKTEST_START} a {BACKTEST_END}")

history.head()

## 1. Feature Engineering

Features inspirees de QC-Py-18 (Feature Engineering):
- **Price-based**: returns 5d/10d/20d, volatility
- **Indicator-based**: RSI, MA ratio, position in range, MACD, ATR

In [ ]:
# Pipeline de feature engineering
def calculate_rsi(prices, period=14):
    """Calcule le RSI"""
    delta = prices.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(lower=0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs = avg_gain / avg_loss.replace(0, 1e-10)
    rsi = 100 - (100 / (1 + rs))
    return rsi

def calculate_features(df, symbol_id):
    """Calcule les features ML pour un secteur"""
    features = pd.DataFrame(index=df.index)
    
    # Price features
    features['returns_5d'] = df['close'].pct_change(5)
    features['returns_10d'] = df['close'].pct_change(10)
    features['returns_20d'] = df['close'].pct_change(20)
    
    # Volatilite
    daily_returns = df['close'].pct_change()
    features['volatility'] = daily_returns.rolling(20).std()
    
    # RSI normalise
    rsi = calculate_rsi(df['close'])
    features['rsi_normalized'] = (rsi - 50) / 50
    
    # MA ratio
    sma_short = df['close'].rolling(20).mean()
    sma_long = df['close'].rolling(50).mean()
    features['ma_ratio'] = sma_short / sma_long
    
    # Position dans le range 20D
    high_20d = df['high'].rolling(20).max()
    low_20d = df['low'].rolling(20).min()
    features['position_in_range'] = (df['close'] - low_20d) / (high_20d - low_20d)
    
    # MACD normalise
    ema_short = df['close'].ewm(span=12).mean()
    ema_long = df['close'].ewm(span=26).mean()
    macd = ema_short - ema_long
    features['macd_normalized'] = macd / df['close']
    
    # Distance from high/low
    features['dist_from_high'] = (high_20d - df['close']) / df['close']
    features['dist_from_low'] = (df['close'] - low_20d) / df['close']
    
    # ATR normalise
    atr = (df['high'] - df['low']).rolling(14).mean()
    features['atr_normalized'] = atr / df['close']
    
    return features

# Calculer les features pour tous les secteurs
all_features = {}
price_data = history['close'].unstack(level=0)
ohlc_data = history

for symbol in symbols:
    symbol_id = symbol.ID
    if symbol_id not in price_data.columns:
        continue
    
    # Extraire OHLC pour ce symbole
    symbol_ohlc = ohlc_data.loc[:, (slice(None), symbol_id)].droplevel(level=0, axis=1)
    
    # Calculer features
    features = calculate_features(symbol_ohlc, symbol_id)
    all_features[symbol_id] = features

print(f"Features calculees pour {len(all_features)} secteurs")

## 2. Creation des Labels (3 Classes)

Labels bases sur les rendements futurs (20 jours):
- **BUY (2)**: Rendement > 2%
- **HOLD (1)**: Rendement entre -1% et 2%
- **AVOID (0)**: Rendement < -1%

In [ ]:
# Creer les labels pour le training
LOOKAHEAD_DAYS = 20
BUY_THRESHOLD = 0.02   # 2%
AVOID_THRESHOLD = -0.01 # -1%

def create_labels(features_dict, price_data, lookahead=20):
    """Cree les labels 3-classes pour tous les secteurs"""
    labels_dict = {}
    
    for symbol_id, features in features_dict.items():
        if symbol_id not in price_data.columns:
            continue
        
        # Rendements futurs
        prices = price_data[symbol_id]
        future_returns = prices.shift(-lookahead) / prices - 1
        
        # Aligner avec features
        labels = pd.Series(index=features.index, dtype=float)
        
        for date in features.index:
            if date in future_returns.index:
                ret = future_returns.loc[date]
                if pd.notna(ret):
                    if ret > BUY_THRESHOLD:
                        labels.loc[date] = 2  # BUY
                    elif ret < AVOID_THRESHOLD:
                        labels.loc[date] = 0  # AVOID
                    else:
                        labels.loc[date] = 1  # HOLD
        
        labels_dict[symbol_id] = labels
    
    return labels_dict

# Creer les labels
labels = create_labels(all_features, price_data, LOOKAHEAD_DAYS)

# Verifier la distribution des classes
all_labels = []
for symbol_labels in labels.values():
    all_labels.extend(symbol_labels.dropna().values)

label_counts = pd.Series(all_labels).value_counts().sort_index()
print("\nDistribution des classes:")
print(f"AVOID (0): {label_counts.get(0, 0)} ({label_counts.get(0, 0)/len(all_labels)*100:.1f}%)")
print(f"HOLD (1): {label_counts.get(1, 0)} ({label_counts.get(1, 0)/len(all_labels)*100:.1f}%)")
print(f"BUY (2): {label_counts.get(2, 0)} ({label_counts.get(2, 0)/len(all_labels)*100:.1f}%)")
print(f"Total: {len(all_labels)} echantillons")

## 3. Preparation des Donnees de Training

- Periode training: 2015-2022
- Features: 7 indicateurs techniques
- Scaler: StandardScaler

In [ ]:
# Preparer les donnees de training
feature_names = [
    'returns_5d', 'returns_10d', 'returns_20d',
    'volatility', 'rsi_normalized', 'ma_ratio', 'position_in_range'
]

X_list = []
y_list = []

# Pour chaque secteur
for symbol_id in all_features.keys():
    features = all_features[symbol_id]
    label = labels[symbol_id]
    
    # Filtrer periode de training
    mask = (features.index >= TRAIN_START) & (features.index <= TRAIN_END)
    
    # Extraire features et labels
    X_symbol = features.loc[mask, feature_names].values
    y_symbol = label.loc[mask].values
    
    # Filtrer NaN
    valid_mask = ~(np.isnan(X_symbol).any(axis=1) | np.isnan(y_symbol))
    X_list.extend(X_symbol[valid_mask])
    y_list.extend(y_symbol[valid_mask])

# Convertir en arrays
X = np.array(X_list)
y = np.array(y_list)

print(f"\nDonnees de training:")
print(f"  Samples: {X.shape[0]}")
print(f"  Features: {X.shape[1]}")
print(f"  Classes: {len(np.unique(y))}")

# Scaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nScaler fitte sur {X.shape[0]} samples")

## 4. Training du Random Forest

Parametres:
- n_estimators: 100
- max_depth: 5
- min_samples_split: 10
- random_state: 42

In [ ]:
# Training Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_scaled, y)

print("Random Forest entraine avec succes")
print(f"\nAccuracy training: {rf.score(X_scaled, y):.2%}")

# Feature importance
importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=True)

print("\nFeature Importance:")
print(importance)

In [ ]:
# Visualiser feature importance
plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'])
plt.title('Feature Importance - Random Forest (Sector Classification)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 5. Backtest Simulation (2023-2025)

Simulation de la strategie:
1. Chaque mois, predire la classe pour chaque secteur
2. Selectionner les top-3 secteurs classes BUY
3. Calculer les returns de la strategie equal-weight

In [ ]:
# Simulation backtest
from pandas.tseries.offsets import MonthBegin

# Dates de rebalancing (1er jour de chaque mois)
rebalance_dates = pd.date_range(BACKTEST_START, BACKTEST_END, freq='MS')

strategy_returns = []
spy_returns = []
sector_predictions = {}

for i, rebalance_date in enumerate(rebalance_dates):
    # Prediction pour chaque secteur
    predictions = {}
    
    for symbol_id in all_features.keys():
        features = all_features[symbol_id]
        
        # Trouver la feature la plus proche
        if rebalance_date in features.index:
            X_new = features.loc[[rebalance_date], feature_names].values
        elif rebalance_date < features.index[-1]:
            # Utiliser la dernière date disponible
            last_date = features.index[features.index <= rebalance_date][-1]
            X_new = features.loc[[last_date], feature_names].values
        else:
            continue
        
        # Predire
        if not np.isnan(X_new).any():
            X_scaled = scaler.transform(X_new)
            pred = rf.predict(X_scaled)[0]
            predictions[symbol_id] = pred
    
    # Top-3 secteurs BUY
    buy_sectors = [s for s, p in predictions.items() if p == 2]
    
    # Si moins de 3 BUY, ajouter HOLD
    if len(buy_sectors) < 3:
        hold_sectors = [s for s, p in predictions.items() if p == 1]
        buy_sectors.extend(hold_sectors[:3 - len(buy_sectors)])
    
    selected = buy_sectors[:3] if len(buy_sectors) >= 3 else buy_sectors
    
    # Calculer returns du mois suivant
    next_month = rebalance_date + pd.DateOffset(months=1)
    
    if len(selected) > 0:
        # Returns de la strategie (equal-weight)
        monthly_returns = []
        for symbol_id in selected:
            prices = price_data[symbol_id]
            if rebalance_date in prices.index and next_month in prices.index:
                ret = prices.loc[next_month] / prices.loc[rebalance_date] - 1
                monthly_returns.append(ret)
        
        if monthly_returns:
            strategy_returns.append(np.mean(monthly_returns))
    
    # Return SPY (benchmark)
    spy_prices = price_data[spy.ID]
    if rebalance_date in spy_prices.index and next_month in spy_prices.index:
        spy_ret = spy_prices.loc[next_month] / spy_prices.loc[rebalance_date] - 1
        spy_returns.append(spy_ret)
    
    sector_predictions[rebalance_date] = {
        'selected': selected,
        'n_buy': len([s for s, p in predictions.items() if p == 2])
    }

print(f"\nBacktest simulation completee")
print(f"  Periode: {BACKTEST_START} a {BACKTEST_END}")
print(f"  Rebalancements: {len(strategy_returns)} mois")

In [ ]:
# Analyser les performances
if len(strategy_returns) > 0 and len(spy_returns) > 0:
    strategy_returns = pd.Series(strategy_returns, index=rebalance_dates[:len(strategy_returns)])
    spy_returns = pd.Series(spy_returns, index=rebalance_dates[:len(spy_returns)])
    
    # Returns cumules
    strategy_cumret = (1 + strategy_returns).cumprod() - 1
    spy_cumret = (1 + spy_returns).cumprod() - 1
    
    # Metriques
    strategy_ann_ret = strategy_returns.mean() * 12
    spy_ann_ret = spy_returns.mean() * 12
    
    strategy_vol = strategy_returns.std() * np.sqrt(12)
    spy_vol = spy_returns.std() * np.sqrt(12)
    
    strategy_sharpe = strategy_ann_ret / strategy_vol if strategy_vol > 0 else 0
    spy_sharpe = spy_ann_ret / spy_vol if spy_vol > 0 else 0
    
    # Drawdown
    strategy_cummax = strategy_cumret.cummax()
    strategy_dd = strategy_cumret - strategy_cummax
    max_dd = strategy_dd.min()
    
    print("\n" + "="*50)
    print("PERFORMANCES BACKTEST (2023-2025)")
    print("="*50)
    print(f"\nStrategy (Sector ML Rotation):")
    print(f"  Return Annualise: {strategy_ann_ret:.2%}")
    print(f"  Volatilite: {strategy_vol:.2%}")
    print(f"  Sharpe Ratio: {strategy_sharpe:.3f}")
    print(f"  Max Drawdown: {max_dd:.2%}")
    print(f"  Total Return: {strategy_cumret.iloc[-1]:.2%}")
    
    print(f"\nBenchmark (SPY):")
    print(f"  Return Annualise: {spy_ann_ret:.2%}")
    print(f"  Volatilite: {spy_vol:.2%}")
    print(f"  Sharpe Ratio: {spy_sharpe:.3f}")
    print(f"  Total Return: {spy_cumret.iloc[-1]:.2%}")
    
    print(f"\nSharpe Ratio Improvement: {(strategy_sharpe - spy_sharpe):.3f}")
    print(f"Outperformance: {(strategy_cumret.iloc[-1] - spy_cumret.iloc[-1]):.2%}")

In [ ]:
# Visualiser equity curve
if len(strategy_returns) > 0:
    plt.figure(figsize=(12, 6))
    plt.plot(strategy_cumret.index, strategy_cumret.values * 100, label='Sector ML Rotation', linewidth=2)
    plt.plot(spy_cumret.index, spy_cumret.values * 100, label='SPY Benchmark', linewidth=2, linestyle='--')
    plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    plt.title('Equity Curve - Sector ML Classification (2023-2025)', fontsize=14, fontweight='bold')
    plt.ylabel('Total Return (%)', fontsize=12)
    plt.xlabel('Date', fontsize=12)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Analyser la distribution des predictions par mois
n_buy_per_month = [v['n_buy'] for v in sector_predictions.values()]

plt.figure(figsize=(10, 5))
plt.hist(n_buy_per_month, bins=range(0, 13), edgecolor='black', alpha=0.7)
plt.xlabel('Nombre de secteurs classes BUY', fontsize=12)
plt.ylabel('Nombre de mois', fontsize=12)
plt.title('Distribution des predictions BUY par mois (2023-2025)', fontsize=14, fontweight='bold')
plt.xticks(range(0, 13))
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"\nStatistiques BUY predictions:")
print(f"  Moyenne: {np.mean(n_buy_per_month):.1f} secteurs/mois")
print(f"  Min: {min(n_buy_per_month)}")
print(f"  Max: {max(n_buy_per_month)}")

## 6. Analyse des Secteurs Selectionnes

Quels secteurs sont le plus souvent selectionnes?

In [ ]:
# Compter les selections par secteur
sector_counts = {}
ticker_to_name = SECTOR_ETFS

for predictions in sector_predictions.values():
    for symbol_id in predictions['selected']:
        # Trouver le ticker
        for ticker, name in SECTOR_ETFS.items():
            # symbol_id contient le ticker
            if ticker in str(symbol_id):
                sector_name = name
                break
        else:
            sector_name = str(symbol_id)
        
        sector_counts[sector_name] = sector_counts.get(sector_name, 0) + 1

# Trier par frequence
sector_counts_sorted = dict(sorted(sector_counts.items(), key=lambda x: x[1], reverse=True))

print("\nFrequence de selection par secteur:")
for sector, count in sector_counts_sorted.items():
    print(f"  {sector}: {count} mois ({count/len(sector_predictions)*100:.1f}%)")

In [ ]:
# Visualiser la frequence de selection
if sector_counts:
    plt.figure(figsize=(10, 6))
    sectors = list(sector_counts_sorted.keys())
    counts = list(sector_counts_sorted.values())
    
    colors = plt.cm.viridis(np.linspace(0, 1, len(sectors)))
    plt.barh(sectors, counts, color=colors, edgecolor='black')
    plt.xlabel('Nombre de selections (mois)', fontsize=12)
    plt.ylabel('Secteur', fontsize=12)
    plt.title('Frequence de Selection par Secteur (2023-2025)', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

## Synthese et Recommandations

### Resultats de la recherche

Cette recherche a demontre la faisabilite d'une approche ML pour la rotation sectorielle:

1. **Feature Engineering**: 11 features inspirees de QC-Py-18
2. **Classification 3-classes**: BUY/HOLD/AVOID avec Random Forest
3. **Selection top-3**: Rebalancement mensuel sur secteurs classes BUY

### Points forts
- Approche systematique et reproductible
- Features techniques bien connues (RSI, MA ratio, etc.)
- Classification 3-classes plus nuance que binaire

### Limitations
- Model static (pas de retraining)
- Features techniques uniquement (pas de macro)
- Equal-weight (pas d'optimisation de portefeuille)

### Ameliorations possibles
- Ajouter features macro (taux d'interet, inflation)
- Retraining annuel du modele
- Portfolio optimisation (QC-Py-21)
- Regime detection (QC-Py-28)